# Roadmap y Guía de Arquitectura: Proyecto Instinct

Este documento contiene la planificación, arquitectura y hoja de ruta para el desarrollo del proyecto **Instinct** en Python para la asignatura de **Programación Orientada a Objetos (POO)**.

---

## 1. Mapa de Módulos del Proyecto

El sistema se divide en **4 capas o componentes independientes**:

```text
+-----------------------------------------------------------------------+
|                           4. Interfaz Visual                          |
|         (Pygame / CustomTkinter / Arcade / PyQt / Console)            |
+-----------------------------------------------------------------------+
                                    |
                                    v
+-----------------------------------------------------------------------+
|                          2. Motor de Simulación                       |
|           (Mundo, Grilla, Casillas, Ciclo de Ticks, Leyes)             |
+-----------------------------------------------------------------------+
            ^                                       ^
            |                                       |
+-----------------------+               +-------------------------------+
|     1. Intérprete     |               |     3. Cargador de Recursos   |
|   (Lexer, Parser, AST,|               |   (Lectura de .te, .ob, .map) |
| Entorno/Percepciones) |               +-------------------------------+
+-----------------------+



## 2. Requerimientos de Librerías (Opciones de Stack)

El proyecto base se puede realizar únicamente con la **Biblioteca Estándar de Python** (salvo para la interfaz gráfica)[cite: 1].

### Opción A: Stack Recomendado (Equilibrio Simplicidad / Presentación)
* **GUI / Renderizado del Mapa:** `pygame` (o `pygame-ce`). Ideal para manejar la grilla 2D, sprites, control de tiempo (FPS) y eventos de ratón.
* **Widgets de Interfaz:** `pygame-gui` o un panel lateral nativo dibujado en Pygame.
* **Tipado y Calidad de Código:** `mypy` y `flake8` (ayudan a mantener la calidad del código para la defensa presencial)[cite: 1].

### Opción B: Stack Módulo Separado (CustomTkinter + Canvas)
* **GUI:** `customtkinter` o `tkinter` estándar.
* **Ventajas:** Facilita la creación de paneles laterales, listas desplegables (`ComboBox` para mapas y criaturas), consolas de logs y barras deslizantes de velocidad. El mapa se renderiza sobre un componente `Canvas`.

## 3. Jerarquía de Clases y Arquitectura POO

Para obtener la máxima calificación, **se debe evitar el uso de cadenas de `if/elif` extensas** (ej. `if accion == "move": ...`)[cite: 1]. Se aplicarán patrones de diseño como **Command**, **Visitor** y **Strategy**[cite: 1].

### A. Capa del Lenguaje e Intérprete (AST)
* **`Lexer` / `Tokenizer`:** Lee el archivo `.ins` línea por línea, elimina comentarios (`#`) e identifica los tokens básicos[cite: 1].
* **`Parser`:** Revisa la sintaxis y construye el Árbol de Sintaxis Abstracta (AST) y la tabla de etiquetas (`labels`)[cite: 1].
* **`Node` (Clase Abstracta base de AST):**
  * **`ExpressionNode`:** Nodos para literales, variables, operaciones binarias (`+`, `-`, `<`, `and`) y llamadas a funciones (`see`, `name`)[cite: 1]. Implementan el método `.evaluate(context)`[cite: 1].
* **`Action` (Clase Abstracta - Patrón Command):**
  * **Subclases:** `MoveAction`, `AttackAction`, `ConsumeAction`, `ReproduceAction`, `RoarAction`, `SayAction`, `WaitAction`[cite: 1].
  * Cada una implementa `.execute(creature, world)`[cite: 1].
* **`Environment` / Contexto:** Maneja la memoria de la criatura (variables locales) y resuelve las percepciones dinámicas (`health`, `enemy_dist`, `random`, etc.) consultando al mundo[cite: 1].

### B. Capa del Mundo
* **`World` / `Grid`:** Representa la grilla de $N \times M$ casillas[cite: 1]. Lleva el control del `tick` actual y procesa la fase de actualización al final de cada turno[cite: 1].
* **`Tile`:** Contiene un `Terrain` obligatoriamente y opcionalmente una entidad (`Object` o `Creature`)[cite: 1].
* **`WorldEntity` (Clase Abstracta):**
  * **`Terrain`:** (`resource_max`, `regen`, `current_resource`)[cite: 1].
  * **`GameObject`:** (`resource_max`, `current_resource`)[cite: 1].
  * **`Creature`:** Posee el AST, atributos de cabecera, estado de vida/edad, posición `x, y` y puntero de instrucción[cite: 1].

  ## 4. Roadmap de Desarrollo Paso a Paso

### Fase 1: Intérprete y Parser del Lenguaje (`.ins`)
- [ ] Implementar la lectura y validación de la cabecera obligatoria (`creature`, `faction`, `health`, `vision`, `lifespan`)[cite: 1].
- [ ] Diseñar el parser de expresiones respetando la precedencia de operadores[cite: 1].
- [ ] Construir la tabla de etiquetas para saltos condicionales e incondicionales (`goto` / `if expr goto`)[cite: 1].
- [ ] Implementar el reporte de errores en tiempo de compilación con número de línea[cite: 1].

### Fase 2: Motor de Simulación en Consola
- [ ] Crear los cargadores de archivos `.te` (terrenos), `.ob` (objetos) y `.map` (mapas)[cite: 1].
- [ ] Implementar las reglas básicas del `World`[cite: 1]:
  * Sorteo aleatorio de turnos por tick[cite: 1].
  * Actualización de percepciones antes de cada turno[cite: 1].
  * Límite de 100 líneas ejecutadas por turno[cite: 1].
  * Aplicación de leyes naturales al final del tick (-1 vida, +1 edad, regeneración de terrenos)[cite: 1].
- [ ] **Prueba de integración:** Ejecutar los ejemplos base (`hobbit.ins`, `ent.ins`, `dwarf.ins`) mediante una representación en texto plano dentro de la terminal[cite: 1].

### Fase 3: Aplicación Visual (GUI)
- [ ] **Fase de Carga:** Mostrar el registro de logs con los archivos cargados correctamente o con errores[cite: 1].
- [ ] **Fase de Preparación:**
  * Selector de mapa disponible[cite: 1].
  * Colocación y eliminación manual de criaturas en casillas válidas[cite: 1].
- [ ] **Fase de Simulación:**
  * Controles de tiempo: Play, Pause, Paso a paso (+1 tick), Reset[cite: 1].
  * Selector de velocidad (2, 10 y 30 ticks por segundo)[cite: 1].
  * Consola gráfica para visualizar rugidos (`roar`), diálogos (`say`) y errores de ejecución[cite: 1].

### Fase 4: Extensiones recomendadas
- [ ] **Inspector de Criaturas:** Panel para inspeccionar variables, vida, edad y línea de código actual al hacer clic sobre una criatura[cite: 1].
- [ ] **Nuevos elementos:** Terrenos o elementos con efectos (lava, agua, trampas)[cite: 1].
- [ ] **Sintaxis extendida:** Soporte para estructuras de control avanzadas (`if/else`, `while`) o nuevas acciones (`shoot`, `heal`)[cite: 1].
- [ ] **Persistencia:** Guardado de simulación y reproducción mediante semillas aleatorias[cite: 1].

---

## 5. Criterios para la Defensa Presencial

1. **Desacoplamiento:** El motor de simulación debe poder funcionar de forma independiente a la interfaz gráfica[cite: 1].
2. **Dominio del código:** Preparación para justificar el uso de patrones de diseño, evaluación de árboles AST y cálculo de percepciones de distancia (distancia Chebyshev)[cite: 1].
3. **Manejo de excepciones:** Los errores de ejecución en una criatura individual deben ser capturados y registrados sin detener la ejecución de la aplicación[cite: 1].